# SLA Breach Prediction — Manual Training Walkthrough
**Dataset:** BPI Challenge 2014 (Rabobank)
**Target:** SLA Breach (binary)
**Model:** XGBoost (primary)

## Step 1: Load the Data

In [ ]:
import pandas as pd
import numpy as np

# Load all three BPI 2014 files
inc = pd.read_csv('bpi2014/Detail_Incident.csv', sep=';', encoding='latin1')
act = pd.read_csv('bpi2014/Detail_Incident_Activity.csv', sep=';', encoding='latin1')
intr = pd.read_csv('bpi2014/Detail_Interaction.csv', sep=';', encoding='latin1')

print(f'Incidents:    {len(inc):,} rows')
print(f'Activities:   {len(act):,} rows')
print(f'Interactions: {len(intr):,} rows')

inc.head()

## Step 2: Explore the Data

In [ ]:
# What do we have?
print('Columns:', list(inc.columns[:20]))
print()
print('Priority distribution:')
print(inc['Priority'].value_counts().sort_index())
print()
print('Category distribution:')
print(inc['Category'].value_counts())
print()
print('Reassignment stats:')
print(f"  Mean: {inc['# Reassignments'].mean():.2f}")
print(f"  Cases with >=1 reassignment: {(inc['# Reassignments']>=1).mean():.1%}")

## Step 3: Clean & Parse

In [ ]:
# Fix Handle Time — uses comma as decimal in the CSV
inc['Handle_Time_Hours'] = pd.to_numeric(
    inc['Handle Time (Hours)'].astype(str).str.replace(',', '.'),
    errors='coerce'
)

# Parse timestamps
for col in ['Open Time', 'Resolved Time', 'Close Time']:
    inc[col] = pd.to_datetime(inc[col], format='mixed', dayfirst=False, errors='coerce')

act['DateStamp'] = pd.to_datetime(act['DateStamp'], format='mixed', dayfirst=True, errors='coerce')

print(f'Handle Time: mean={inc["Handle_Time_Hours"].mean():.1f}h, median={inc["Handle_Time_Hours"].median():.1f}h')
print(f'Open Time range: {inc["Open Time"].min()} to {inc["Open Time"].max()}')

## Step 4: Define the Target — SLA Breach
We define breach as exceeding **2× the median handle time** for that priority level.
This uses the data's own operational norms rather than an arbitrary threshold.

In [ ]:
# Median handle time per priority
priority_medians = inc.groupby('Priority')['Handle_Time_Hours'].median()
print('Median handle time by priority:')
print(priority_medians)

# SLA threshold = 2x median
sla_thresholds = {p: m * 2.0 for p, m in priority_medians.items()}
inc['SLA_Threshold'] = inc['Priority'].map(sla_thresholds)
inc['SLA_Breached'] = (inc['Handle_Time_Hours'] > inc['SLA_Threshold']).astype(int)

df = inc[inc['Handle_Time_Hours'].notna() & inc['Priority'].notna()].copy()
print(f'\nSLA breach rate: {df["SLA_Breached"].mean():.1%}')
print(df['SLA_Breached'].value_counts())

## Step 5: Feature Engineering
Extract meaningful features from the raw data.

In [ ]:
# --- From Activity Log ---
assign_events = act[act['IncidentActivity_Type'].isin(['Assignment', 'Reassignment'])]

# First assignment group
first_assign = (assign_events.sort_values('DateStamp')
                .groupby('Incident ID')['Assignment Group'].first()
                .reset_index()
                .rename(columns={'Assignment Group': 'First_Assignment_Group'}))
df = df.merge(first_assign, on='Incident ID', how='left')

# Groups touched
groups_touched = (assign_events.groupby('Incident ID')['Assignment Group']
                  .nunique().reset_index()
                  .rename(columns={'Assignment Group': 'Num_Groups_Touched'}))
df = df.merge(groups_touched, on='Incident ID', how='left')
df['Num_Groups_Touched'] = df['Num_Groups_Touched'].fillna(0).astype(int)

# Total events per incident
event_counts = act.groupby('Incident ID').size().reset_index(name='Total_Activity_Events')
df = df.merge(event_counts, on='Incident ID', how='left')
df['Total_Activity_Events'] = df['Total_Activity_Events'].fillna(0).astype(int)

# Assignment delay
first_assign_time = (assign_events.sort_values('DateStamp')
                     .groupby('Incident ID')['DateStamp'].first()
                     .reset_index()
                     .rename(columns={'DateStamp': 'First_Assignment_Time'}))
df = df.merge(first_assign_time, on='Incident ID', how='left')
df['Assignment_Delay_Hours'] = ((df['First_Assignment_Time'] - df['Open Time'])
                                 .dt.total_seconds() / 3600).clip(lower=0).fillna(0)

print(f'Features from activity log: done')
print(f'  Assignment delay: mean={df["Assignment_Delay_Hours"].mean():.2f}h')
print(f'  Groups touched: mean={df["Num_Groups_Touched"].mean():.2f}')

In [ ]:
# --- Group performance features ---
group_stats = df.groupby('First_Assignment_Group').agg(
    Group_Case_Count=('Incident ID', 'count'),
    Group_Avg_Handle_Time=('Handle_Time_Hours', 'mean'),
    Group_Breach_Rate=('SLA_Breached', 'mean'),
    Group_Reassignment_Rate=('# Reassignments', lambda x: (x > 0).mean()),
).reset_index()
df = df.merge(group_stats, on='First_Assignment_Group', how='left')

# --- Interaction features ---
intr_agg = intr.groupby('Related Incident').agg(
    Interaction_Count=('Interaction ID', 'count'),
    FCR_Rate=('First Call Resolution', lambda x: (x == 'Y').mean()),
).reset_index()
df = df.merge(intr_agg, left_on='Incident ID', right_on='Related Incident', how='left')
df['Interaction_Count'] = df['Interaction_Count'].fillna(0).astype(int)
df['FCR_Rate'] = df['FCR_Rate'].fillna(0)

# --- Temporal features ---
df['Open_Hour'] = df['Open Time'].dt.hour
df['Open_DayOfWeek'] = df['Open Time'].dt.dayofweek
df['Is_Weekend'] = (df['Open_DayOfWeek'] >= 5).astype(int)

# --- Case complexity ---
df['Num_Related_Incidents'] = pd.to_numeric(df['# Related Incidents'], errors='coerce').fillna(0)
df['Has_Reopen'] = df['Reopen Time'].notna().astype(int)

# --- Encode categoricals ---
from sklearn.preprocessing import LabelEncoder
df['CI_Type_Encoded'] = LabelEncoder().fit_transform(df['CI Type (aff)'].fillna('unknown').astype(str))
df['Category_Encoded'] = LabelEncoder().fit_transform(df['Category'].fillna('unknown').astype(str))

print('All features engineered!')
print(f'Dataset shape: {df.shape}')

## Step 6: Prepare Feature Matrix

In [ ]:
# Define feature sets
FEATURES_5 = ['Priority', 'Impact', 'Urgency', '# Reassignments', 'Num_Related_Incidents']

FEATURES_ALL = [
    'Priority', 'Impact', 'Urgency', '# Reassignments', 'Num_Related_Incidents',
    'Open_Hour', 'Open_DayOfWeek', 'Is_Weekend',
    'Assignment_Delay_Hours', 'Num_Groups_Touched', 'Total_Activity_Events',
    'Group_Avg_Handle_Time', 'Group_Breach_Rate', 'Group_Reassignment_Rate',
    'Group_Case_Count', 'Interaction_Count', 'FCR_Rate',
    'Has_Reopen', 'CI_Type_Encoded', 'Category_Encoded',
]

# Drop NaN rows
df_model = df[FEATURES_ALL + ['SLA_Breached']].dropna()
X = df_model[FEATURES_ALL]
y = df_model['SLA_Breached']

print(f'Final dataset: {X.shape[0]:,} rows x {X.shape[1]} features')
print(f'Breach rate: {y.mean():.1%}')
print(f'\nFeatures used:')
for i, f in enumerate(FEATURES_ALL, 1):
    print(f'  {i:2d}. {f}')

## Step 7: Train-Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f'Train: {len(X_train):,} rows')
print(f'Test:  {len(X_test):,} rows')
print(f'Train breach rate: {y_train.mean():.1%}')
print(f'Test breach rate:  {y_test.mean():.1%}')

## Step 8: Train XGBoost (manually, step by step)

In [ ]:
from xgboost import XGBClassifier

# Create the model with hyperparameters
xgb_model = XGBClassifier(
    n_estimators=300,       # 300 trees
    max_depth=6,            # each tree can be 6 levels deep
    learning_rate=0.05,     # each tree contributes only 5% — slow and steady
    subsample=0.8,          # each tree sees 80% of rows (randomness)
    colsample_bytree=0.8,   # each tree sees 80% of columns (randomness)
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),  # handle class imbalance
    eval_metric='logloss',  # binary cross-entropy loss
    random_state=42,
    verbosity=0,
)

# Train it
print('Training XGBoost...')
xgb_model.fit(X_train, y_train)
print('Done!')
print(f'Number of trees built: {xgb_model.n_estimators}')
print(f'Features used: {xgb_model.n_features_in_}')

## Step 9: Make Predictions

In [ ]:
# Predict labels (0 or 1)
y_pred = xgb_model.predict(X_test)

# Predict probabilities (0.0 to 1.0)
y_proba = xgb_model.predict_proba(X_test)[:, 1]  # column 1 = probability of breach

# Show a few examples
print('Sample predictions (first 10 test cases):')
print(f'{"Actual":>8s}  {"Predicted":>10s}  {"Probability":>12s}')
for actual, pred, prob in zip(y_test.head(10), y_pred[:10], y_proba[:10]):
    marker = 'CORRECT' if actual == pred else 'WRONG'
    print(f'{actual:>8d}  {pred:>10d}  {prob:>12.4f}  {marker}')

## Step 10: Evaluate — All Metrics

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report)

print('=== CLASSIFICATION REPORT ===')
print(classification_report(y_test, y_pred, target_names=['No Breach', 'SLA Breach']))

print('=== INDIVIDUAL METRICS ===')
print(f'Accuracy:  {accuracy_score(y_test, y_pred):.4f}')
print(f'Precision: {precision_score(y_test, y_pred):.4f}')
print(f'Recall:    {recall_score(y_test, y_pred):.4f}')
print(f'F1 Score:  {f1_score(y_test, y_pred):.4f}')
print(f'ROC-AUC:   {roc_auc_score(y_test, y_proba):.4f}')

print('\n=== CONFUSION MATRIX ===')
cm = confusion_matrix(y_test, y_pred)
print(f'  TN (correct no-breach) = {cm[0,0]}')
print(f'  FP (false alarm)       = {cm[0,1]}')
print(f'  FN (missed breach)     = {cm[1,0]}')
print(f'  TP (caught breach)     = {cm[1,1]}')

## Step 11: 5-Fold Cross Validation

In [ ]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold_scores = []
for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):
    # Split
    Xf_train = X_train.iloc[train_idx]
    yf_train = y_train.iloc[train_idx]
    Xf_val   = X_train.iloc[val_idx]
    yf_val   = y_train.iloc[val_idx]
    
    # Train a fresh model on this fold
    fold_model = XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=(yf_train == 0).sum() / (yf_train == 1).sum(),
        eval_metric='logloss', random_state=42, verbosity=0,
    )
    fold_model.fit(Xf_train, yf_train)
    
    # Score
    fold_proba = fold_model.predict_proba(Xf_val)[:, 1]
    fold_auc = roc_auc_score(yf_val, fold_proba)
    fold_scores.append(fold_auc)
    print(f'  Fold {fold}: AUC = {fold_auc:.4f}')

print(f'\n  Mean AUC: {np.mean(fold_scores):.4f} +/- {np.std(fold_scores):.4f}')

## Step 12: ROC Curve

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve

fpr, tpr, thresholds = roc_curve(y_test, y_proba)
auc = roc_auc_score(y_test, y_proba)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, label=f'XGBoost (AUC = {auc:.3f})', linewidth=2)
plt.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random (AUC = 0.500)')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve — SLA Breach Prediction')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Step 13: Feature Importance

In [ ]:
# Built-in feature importance (how often each feature was used in splits)
importances = xgb_model.feature_importances_
sorted_idx = np.argsort(importances)[::-1]

print('Feature Importance (top 15):')
for i in sorted_idx[:15]:
    print(f'  {FEATURES_ALL[i]:30s}  {importances[i]:.4f}')

plt.figure(figsize=(10, 6))
plt.barh(range(len(sorted_idx)), importances[sorted_idx][::-1])
plt.yticks(range(len(sorted_idx)), [FEATURES_ALL[i] for i in sorted_idx][::-1])
plt.xlabel('Importance')
plt.title('XGBoost Feature Importance')
plt.tight_layout()
plt.show()

## Step 14: SHAP Explanation (why each prediction was made)

In [ ]:
import shap

explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test.iloc[:200])  # explain 200 test cases

# Beeswarm plot — shows all features, all cases
shap.summary_plot(shap_values, X_test.iloc[:200], feature_names=FEATURES_ALL)

In [ ]:
# Explain ONE specific case
case_idx = 0
print(f'Case {case_idx}: actual={y_test.iloc[case_idx]}, predicted prob={y_proba[case_idx]:.3f}')
print()
shap.force_plot(explainer.expected_value, shap_values[case_idx], X_test.iloc[case_idx],
                feature_names=FEATURES_ALL, matplotlib=True)

## Step 15: Compare All 4 Models

In [ ]:
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier

models = {
    'XGBoost': XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05,
                             subsample=0.8, colsample_bytree=0.8,
                             scale_pos_weight=(y_train==0).sum()/(y_train==1).sum(),
                             eval_metric='logloss', random_state=42, verbosity=0),
    'LightGBM': LGBMClassifier(n_estimators=300, max_depth=6, learning_rate=0.05,
                               is_unbalance=True, random_state=42, verbose=-1),
    'CatBoost': CatBoostClassifier(iterations=300, depth=6, learning_rate=0.05,
                                   auto_class_weights='Balanced', random_seed=42, verbose=0),
    'RandomForest': RandomForestClassifier(n_estimators=300, max_depth=12,
                                          class_weight='balanced', random_state=42),
}

print(f'{"Model":15s}  {"AUC":>8s}  {"F1":>8s}  {"Precision":>10s}  {"Recall":>8s}')
print('-' * 55)

for name, model in models.items():
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_test)[:, 1]
    pred = model.predict(X_test)
    print(f'{name:15s}  {roc_auc_score(y_test, proba):>8.4f}  '
          f'{f1_score(y_test, pred):>8.4f}  '
          f'{precision_score(y_test, pred):>10.4f}  '
          f'{recall_score(y_test, pred):>8.4f}')